# Conference Demo — Context Engineering for Long-Running Agents\nThis notebook clones the GitHub project, configures secrets, streams the LangGraph run, and displays durable evidence and metrics.

In [ ]:
REPO_URL = "https://github.com/swarnamalya-mohan/context-engineering-long-running-agent.git"\n!git clone -q $REPO_URL\n%cd context-engineering-long-running-agent\n!pip -q install -e .\n

In [ ]:
import os\nfrom google.colab import userdata\nos.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")\ntry:\n    os.environ["GITHUB_TOKEN"] = userdata.get("GITHUB_TOKEN")\nexcept Exception:\n    pass\nos.environ["TARGET_REPOS"] = "30"   # change to 100 for full run\nos.environ["DEEP_REVIEW"] = "10"    # change to 20 for full run\n

In [ ]:
from repo_agent.graph import build_graph\nfrom repo_agent.config import Settings\nfrom rich import print\nsettings = Settings.from_env()\ngraph = build_graph(settings=settings, repo_root='.')\nconfig = {'configurable': {'thread_id': 'conference-demo'}}\nfor update in graph.stream({}, config=config, stream_mode='updates'):\n    print(update)\nstate = graph.get_state(config).values\n

In [ ]:
import pandas as pd\nfrom repo_agent.metrics import candidate_frame, evidence_frame, event_frame\ndisplay(candidate_frame(state).sort_values('score', ascending=False, na_position='last'))\ndisplay(event_frame(state).tail(30))\ndisplay(evidence_frame(state).head(50))\nprint(state.get('metrics', {}))\n

## What to point out live\n1. OpenSpec change event adds enterprise SSO.\n2. Previously reviewed repos are queued again.\n3. SSO skill appears only after that change.\n4. AGPL change reopens stale license rejections.\n5. Compaction shrinks transient history while durable evidence/spec state survives.